## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [49]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool,set_default_openai_client, set_tracing_disabled, set_tracing_export_api_key
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown
from openai import AsyncOpenAI
import json

In [50]:
load_dotenv(override=True)

True

In [51]:
set_tracing_disabled(False)

In [52]:
set_tracing_export_api_key(os.environ.get("OPENAI_API_KEY"))

In [53]:
custom_client = AsyncOpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),  # Your Groq key
    base_url="https://api.groq.com/openai/v1"
)

In [54]:
set_default_openai_client(custom_client)

In [55]:
SEARCH_MODEL = "openai/groq/compound-mini"  
REASONING_MODEL = "llama-3.3-70b-versatile"  

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 2.5 cents per call.

Costs are here: https://platform.openai.com/docs/pricing#web-search

In [56]:
INSTRUCTIONS = """You are a research assistant. Given a search term, provide a ONE SENTENCE summary.
Maximum 15 words. Just the key fact. No bullet points, no explanations."""

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    # tools=[WebSearchTool(search_context_size="low")],
    model=SEARCH_MODEL,
    # model_settings=ModelSettings(tool_choice="required"),
)

In [57]:
message = "whats todays news as of 24 feb 2026"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

Milk contamination kills two, injures many in Andhra Pradesh; NATO marks invasion anniversary.

### As always, take a look at the trace

https://platform.openai.com/traces

### We will now use Structured Outputs, and include a description of the fields

In [58]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 2

INSTRUCTIONS = f"""Given a query, output {HOW_MANY_SEARCHES} search terms to best answer it.
For each search term, provide:
1. The search query
2. A brief reason why this search is important

Format as JSON with this structure:
{{
    "searches": [
        {{
            "query": "search term here",
            "reason": "reason here"
        }}
    ]
}}"""

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model=REASONING_MODEL,
    # output_type=WebSearchPlan,
)

In [59]:

message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

```json
{
    "searches": [
        {
            "query": "2025 AI agent framework comparison",
            "reason": "To find a comprehensive overview of the latest AI agent frameworks available in 2025, including their features, strengths, and weaknesses, and to compare them in order to make an informed decision."
        },
        {
            "query": "State-of-the-art AI agent architectures 2025",
            "reason": "To discover the most advanced and innovative AI agent architectures being developed and used in 2025, and to understand how they can be applied to various industries and applications."
        }
    ]
}
```


In [78]:
@function_tool
def send_email_tool(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("jay.k@crestskillserve.com")
    to_email = To("jaygdsc0@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [81]:
email_agent = Agent(
    name="Email agent",
    instructions="""You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line.""",
    tools=[send_email_tool],
    model=REASONING_MODEL,
)



In [73]:
INSTRUCTIONS = """You are a senior researcher. Based on the research provided, create a report.

Your response MUST be a valid JSON object with exactly these fields:
{
  "short_summary": "2-3 sentence summary",
  "markdown_report": "# Title\\n\\nFull report in markdown...",
  "follow_up_questions": ["Question 1", "Question 2"]
}

IMPORTANT: 
- Return ONLY the JSON object, no other text
- Do NOT wrap in ```json or ``` markers
- The markdown_report should be at least 1000 words"""

writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model=REASONING_MODEL,
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [74]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    
    # Clean the output
    output = result.final_output.strip()
    if output.startswith("```json"):
        output = output[7:]
    elif output.startswith("```"):
        output = output[3:]
    if output.endswith("```"):
        output = output[:-3]
    output = output.strip()
    
    # Parse the JSON
    data = json.loads(output)
    
    # Validate the structure
    if "searches" not in data or not isinstance(data["searches"], list):
        raise ValueError("Invalid search plan format")
    
    # Convert to WebSearchPlan object
    search_plan = WebSearchPlan(**data)
    
    print(f"Will perform {len(search_plan.searches)} searches")
    return search_plan


async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item sequentially with size limits """
    print("Searching...")
    
    results = []
    for i, item in enumerate(search_plan.searches):
        print(f"Search {i+1}/{len(search_plan.searches)}: {item.query}")
        
        # Add LONG delay between searches - 60 seconds to avoid rate limits
        if i > 0:
            print(f"Waiting 60 seconds before next search to avoid rate limits...")
            await asyncio.sleep(60)  # 60 second delay
        
        # Run search with timeout
        try:
            result = await asyncio.wait_for(
                search(item),
                timeout=30.0
            )
            
            results.append(result)
            print(f"  ✓ Completed ({len(result)} chars)")
            
        except Exception as e:
            print(f"  ✗ Error: {str(e)[:50]}")
            results.append("Search failed")
    
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}\nProvide VERY BRIEF summary (max 50 words)."
    
    try:
        result = await Runner.run(search_agent, input)
        output = result.final_output.strip()
        
        # Ensure we don't return empty results
        if not output:
            return f"No results found for: {item.query}"
        
        return output
    except Exception as e:
        print(f"Search error for {item.query}: {str(e)[:100]}")
        return f"Error searching for: {item.query}"

### The next 2 functions write a report and email it

In [82]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    
    # Get the raw output
    output = result.final_output.strip()
    print(f"Raw output preview: {output[:100]}...")
    
    # Try multiple cleaning strategies
    try:
        # Strategy 1: Try direct parse
        data = json.loads(output)
    except:
        try:
            # Strategy 2: Remove markdown code blocks
            cleaned = output
            if "```json" in cleaned:
                cleaned = cleaned.split("```json")[1]
            elif "```" in cleaned:
                cleaned = cleaned.split("```")[1]
            if "```" in cleaned:
                cleaned = cleaned.split("```")[0]
            cleaned = cleaned.strip()
            data = json.loads(cleaned)
        except:
            try:
                # Strategy 3: Find JSON between curly braces
                import re
                json_match = re.search(r'\{.*\}', output, re.DOTALL)
                if json_match:
                    data = json.loads(json_match.group())
                else:
                    raise
            except Exception as e:
                print(f"Failed to parse JSON. Output: {output[:200]}")
                raise
    
    # Convert to ReportData object
    report = ReportData(**data)
    
    print("Finished writing report")
    return report
    
   

async def send_email(report):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Showtime!

In [85]:
query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")


Starting research...
Planning searches...
Will perform 2 searches
Searching...
Search 1/2: Top AI frameworks 2025


Error getting response: Error code: 413 - {'error': {'message': 'Request Entity Too Large', 'type': 'invalid_request_error', 'code': 'request_too_large'}}. (request_id: req_01kj7pk14cfmdskk2dt59qmjxe)


Search error for Top AI frameworks 2025: Error code: 413 - {'error': {'message': 'Request Entity Too Large', 'type': 'invalid_request_error',
  ✓ Completed (43 chars)
Search 2/2: New AI agent architectures 2025
Waiting 60 seconds before next search to avoid rate limits...


Error getting response: Error code: 413 - {'error': {'message': 'Request Entity Too Large', 'type': 'invalid_request_error', 'code': 'request_too_large'}}. (request_id: req_01kj7pn1myf94vdgfh51ryafcw)


Search error for New AI agent architectures 2025: Error code: 413 - {'error': {'message': 'Request Entity Too Large', 'type': 'invalid_request_error',
  ✓ Completed (52 chars)
Finished searching
Thinking about report...
Raw output preview: {
  "short_summary": "The latest AI agent frameworks in 2025 are not available due to search errors....
Finished writing report
Writing email...
Email sent
Hooray!


### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>